# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilip-chendra/FlyRank-Week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

The baseline rule identifies pages that are likely to benefit from a content refresh.

A page receives a higher score when it:
- has high search impressions,
- has older content,
- has a relatively low click-through rate (CTR).

These signals are combined into a simple baseline score to rank pages for manual review. The output is intended for decision support and not as proof that refreshing the page will improve performance.

## Reason Codes

- **stale_visible_page** – The page is old and still receives search visibility.
- **low_ctr_visible_page** – The page receives impressions but has a relatively low CTR.
- **refresh_candidate** – The page has a high overall baseline score and should be reviewed first.


In [6]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

print("Working Directory:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

print("\nContent Age Summary")
print(df["content_age_days"].describe())

print("\nCTR Summary")
print(df["ctr"].describe())

Working Directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Dataset Shape: (30000, 44)

Content Age Summary
count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000
Name: content_age_days, dtype: float64

CTR Summary
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Baseline Ranking

The baseline score combines three measurable signals:

- Search impressions (visibility)
- Content age (staleness)
- Click-through rate (CTR)

Higher scores indicate pages that may deserve a manual content refresh review.

The ranked queue is exported to:

`work/outputs/baseline_action_score.csv`

In [7]:
import os

# Normalize features
df["visibility_score"] = (
    df["impressions_90d"] / df["impressions_90d"].max()
)

df["age_score"] = (
    df["content_age_days"] / df["content_age_days"].max()
)

df["ctr_score"] = (
    1 - (df["ctr"] / df["ctr"].max())
)

# Baseline score
df["baseline_score"] = (
    0.40 * df["visibility_score"] +
    0.40 * df["age_score"] +
    0.20 * df["ctr_score"]
)

# Reason codes
df["reason_code"] = "refresh_candidate"

df.loc[
    (df["content_age_days"] > df["content_age_days"].median()),
    "reason_code"
] = "stale_visible_page"

df.loc[
    (df["ctr"] < df["ctr"].median()),
    "reason_code"
] = "low_ctr_visible_page"

# Action
df["action"] = "Refresh Content"

# Rank
df = df.sort_values(
    "baseline_score",
    ascending=False
)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

df.to_csv(output_path, index=False)

print("CSV saved to:", output_path)

df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(20)

CSV saved to: work/outputs/baseline_action_score.csv


,content_id,baseline_score,reason_code,action
6653,content_5fe46e04994d,0.980571,stale_visible_page,Refresh Content
17812,content_aaef01a50def,0.914635,stale_visible_page,Refresh Content
26844,content_8c19996aa890,0.908764,stale_visible_page,Refresh Content
21819,content_4c36c775b818,0.872588,stale_visible_page,Refresh Content
29879,content_1a9e894be2e2,0.862935,stale_visible_page,Refresh Content
18870,content_db5989a78dd3,0.781825,stale_visible_page,Refresh Content
29400,content_2dba2b1f9536,0.754245,stale_visible_page,Refresh Content
21565,content_9532f197bbc8,0.752753,stale_visible_page,Refresh Content
13537,content_2c2606c5d176,0.724087,stale_visible_page,Refresh Content
19636,content_2cb567c3c89b,0.692867,refresh_candidate,Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The following table shows the highest-ranked pages according to the baseline rule.

Each page includes:
- Recommended action
- Reason code
- Confidence level
- A note describing what could make the recommendation incorrect

These recommendations should be reviewed manually before any content changes are made.

In [8]:
top20 = df[
    [
        "content_id",
        "baseline_score",
        "action",
        "reason_code"
    ]
].head(20).copy()

top20["confidence_note"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "Seasonality, temporary ranking changes, or recent content updates."
)

print(top20)

top20

                 content_id  baseline_score           action  \
6653   content_5fe46e04994d        0.980571  Refresh Content   
17812  content_aaef01a50def        0.914635  Refresh Content   
26844  content_8c19996aa890        0.908764  Refresh Content   
21819  content_4c36c775b818        0.872588  Refresh Content   
29879  content_1a9e894be2e2        0.862935  Refresh Content   
18870  content_db5989a78dd3        0.781825  Refresh Content   
29400  content_2dba2b1f9536        0.754245  Refresh Content   
21565  content_9532f197bbc8        0.752753  Refresh Content   
13537  content_2c2606c5d176        0.724087  Refresh Content   
19636  content_2cb567c3c89b        0.692867  Refresh Content   
22402  content_9463d30d5826        0.688559  Refresh Content   
29137  content_9b934e3e7101        0.662246  Refresh Content   
15998  content_3cc803523902        0.650754  Refresh Content   
29998  content_ab26273a7e7a        0.649630  Refresh Content   
19499  content_0e70a832cb7a        0.649

,content_id,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
6653,content_5fe46e04994d,0.980571,Refresh Content,stale_visible_page,Medium,"Seasonality, temporary ranking changes, or rec..."
17812,content_aaef01a50def,0.914635,Refresh Content,stale_visible_page,Medium,"Seasonality, temporary ranking changes, or rec..."
26844,content_8c19996aa890,0.908764,Refresh Content,stale_visible_page,Medium,"Seasonality, temporary ranking changes, or rec..."
21819,content_4c36c775b818,0.872588,Refresh Content,stale_visible_page,Medium,"Seasonality, temporary ranking changes, or rec..."
29879,content_1a9e894be2e2,0.862935,Refresh Content,stale_visible_page,Medium,"Seasonality, temporary ranking changes, or rec..."
18870,content_db5989a78dd3,0.781825,Refresh Content,stale_visible_page,Medium,"Seasonality, temporary ranking changes, or rec..."
29400,content_2dba2b1f9536,0.754245,Refresh Content,stale_visible_page,Medium,"Seasonality, temporary ranking changes, or rec..."
21565,content_9532f197bbc8,0.752753,Refresh Content,stale_visible_page,Medium,"Seasonality, temporary ranking changes, or rec..."
13537,content_2c2606c5d176,0.724087,Refresh Content,stale_visible_page,Medium,"Seasonality, temporary ranking changes, or rec..."
19636,content_2cb567c3c89b,0.692867,Refresh Content,refresh_candidate,Medium,"Seasonality, temporary ranking changes, or rec..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks and Leakage Check

Some recommendations may not require a content refresh.

Examples include:

- Seasonal content
- Recently updated pages
- Pages affected by temporary search trends
- Low-traffic pages with unstable metrics

To avoid data leakage, future information and label-derived fields were excluded from the baseline score calculation.

The baseline rule only uses information that would be available before making the refresh decision.

In [9]:
print("Leakage Check")

excluded_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("\nExcluded Columns")

for col in excluded_columns:
    print("-", col)

print("\nThese columns were NOT used when calculating the baseline score.")

print("\nPossible Weak Picks")

weak_picks = [
    "Seasonal pages",
    "Recently updated pages",
    "Temporary traffic spikes",
    "Low-volume pages"
]

for item in weak_picks:
    print("-", item)

Leakage Check

Excluded Columns
- trend_direction
- trend_pct
- is_declining_label

These columns were NOT used when calculating the baseline score.

Possible Weak Picks
- Seasonal pages
- Recently updated pages
- Temporary traffic spikes
- Low-volume pages


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card.